In [ ]:
import psutil,time,os,json
from datetime import datetime
from IPython.display import display,HTML

# ==============================================================================
# REAL TIME MEMORY MONITOR
# VS CODE + JUPYTER NOTEBOOK OWNERSHIP ANALYSIS
# KERNEL MAPPING / ORPHAN DETECTION
# ==============================================================================

dashboard_display=display(HTML(""),display_id=True)

# ==============================================================================
# Configuration
# ==============================================================================

JUPYTER_RUNTIME=os.path.join(
    os.environ.get("APPDATA",""),
    "jupyter",
    "runtime"
)

previous_memory={}

# ==============================================================================
# Formatting
# ==============================================================================

def format_memory(value):
    if value>=1024**3:
        return f"{value/1024**3:,.2f} GB"
    return f"{value/1024**2:,.2f} MB"

# ==============================================================================
# Load Jupyter Kernel Metadata
# ==============================================================================

def load_jupyter_kernels():

    kernels={}

    if not os.path.exists(JUPYTER_RUNTIME):
        return kernels

    for file in os.listdir(JUPYTER_RUNTIME):

        if not(file.startswith("kernel-") and file.endswith(".json")):
            continue

        path=os.path.join(
            JUPYTER_RUNTIME,
            file
        )

        try:
            with open(path,"r") as f:
                data=json.load(f)

            kernels[file]={
                "file":file,
                "path":path,
                "created":datetime.fromtimestamp(
                    os.path.getctime(path)
                )
            }

        except:
            pass

    return kernels

# ==============================================================================
# Process Collector
# ==============================================================================

def collect_processes():

    processes=[]

    for p in psutil.process_iter(
        [
            "pid",
            "ppid",
            "name",
            "memory_info",
            "cmdline",
            "create_time"
        ]
    ):

        try:
            processes.append({
                "pid":p.info["pid"],
                "ppid":p.info["ppid"],
                "name":p.info["name"] or "",
                "memory":p.info["memory_info"].rss,
                "cmd":" ".join(p.info["cmdline"] or []),
                "created":datetime.fromtimestamp(
                    p.info["create_time"]
                )
            })

        except:
            pass

    return sorted(
        processes,
        key=lambda x:x["memory"],
        reverse=True
    )

# ==============================================================================
# Classification
# ==============================================================================

def classify(p):

    text=(p["name"]+" "+p["cmd"]).lower()

    if "ipykernel" in text:
        return "JUPYTER KERNEL"

    if "pylance" in text:
        return "PYLANCE"

    if "code.exe" in text:

        if p["ppid"]:
            try:
                parent=psutil.Process(
                    p["ppid"]
                ).name().lower()

                if parent=="explorer.exe":
                    return "VS CODE MAIN"

            except:
                pass

        return "VS CODE CHILD"

    return ""

# ==============================================================================
# Find Kernel PID Matches
# ==============================================================================

def find_kernel_processes(processes):

    return [
        p for p in processes
        if "ipykernel" in p["cmd"].lower()
    ]

# ==============================================================================
# Application Summary
# ==============================================================================

def summary(processes,key):

    total=0
    count=0

    for p in processes:

        text=(p["name"]+" "+p["cmd"]).lower()

        if key in text:
            count+=1
            total+=p["memory"]

    return count,total

# ==============================================================================
# MAIN LOOP
# ==============================================================================

while True:

    processes=collect_processes()
    kernels=load_jupyter_kernels()
    lines=[]

    lines.extend([
        "="*110,
        "                 REAL TIME MEMORY MONITOR",
        "="*110,
        "",
        f"TIME : {datetime.now()}",
        ""
    ])

    # --------------------------------------------------------------------------
    # Memory
    # --------------------------------------------------------------------------

    mem=psutil.virtual_memory()

    lines.extend([
        "SYSTEM MEMORY",
        "-"*110,
        f"Total RAM       : {mem.total/1024**3:.2f} GB",
        f"Used RAM        : {mem.used/1024**3:.2f} GB",
        f"Available RAM   : {mem.available/1024**3:.2f} GB",
        f"Usage           : {mem.percent:.1f}%",
        ""
    ])

    # --------------------------------------------------------------------------
    # Application Summary
    # --------------------------------------------------------------------------

    lines.extend([
        "APPLICATION SUMMARY",
        "-"*110
    ])

    for app in [
        "chrome",
        "code.exe",
        "python",
        "ipykernel"
    ]:

        c,m=summary(
            processes,
            app
        )

        lines.append(
            f"{app.upper():<20}"
            f"Processes:{c:<5}"
            f"Memory:{format_memory(m)}"
        )

    # --------------------------------------------------------------------------
    # Top Consumers
    # --------------------------------------------------------------------------

    lines.extend([
        "",
        "TOP MEMORY CONSUMERS",
        "-"*110,
        f"{'Process':<25}{'PID':<10}{'Memory':>15}{'Change':>15}",
        "-"*110
    ])

    for p in processes[:15]:

        old=previous_memory.get(
            p["pid"],
            p["memory"]
        )

        delta=p["memory"]-old

        lines.append(
            f"{p['name']:<25}"
            f"{p['pid']:<10}"
            f"{format_memory(p['memory']):>15}"
            f"{format_memory(delta):>15}"
        )

    # --------------------------------------------------------------------------
    # Kernel Mapping
    # --------------------------------------------------------------------------

    lines.extend([
        "",
        "JUPYTER NOTEBOOK KERNEL MAP",
        "-"*110
    ])

    kernel_processes=find_kernel_processes(
        processes
    )

    for p in kernel_processes:

        age=datetime.now()-p["created"]
        status="IDLE" if p["memory"]<40*1024**2 else "ACTIVE"

        lines.extend([
            f"Python PID:{p['pid']}   {format_memory(p['memory'])}   {status}",
            f"    Started : {p['created']}",
            f"    Age     : {age}"
        ])

        for k in kernels.values():
            lines.append(
                f"    Kernel  : {k['file']}"
            )

        lines.append("")

    lines.extend([
        "JUPYTER SUMMARY",
        "-"*110,
        f"Detected Kernel Processes : {len(kernel_processes)}",
        f"Runtime Kernel Files      : {len(kernels)}",
        ""
    ])

    # --------------------------------------------------------------------------
    # Orphan Detection
    # --------------------------------------------------------------------------

    lines.extend([
        "KERNEL HEALTH",
        "-"*110
    ])

    for p in kernel_processes:

        age=datetime.now()-p["created"]

        if age.days>1:
            lines.append(
                f"⚠ POSSIBLE OLD KERNEL PID:{p['pid']} Age:{age}"
            )

    if not kernel_processes:
        lines.append(
            "No active kernels detected"
        )

    previous_memory={
        p["pid"]:p["memory"]
        for p in processes
    }

    lines.extend([
        "",
        "="*110,
        "Refreshing every 5 seconds...",
        "="*110
    ])

    dashboard_display.update(
        HTML(
            "<pre>"+
            "\n".join(lines)+
            "</pre>"
        )
    )

    time.sleep(5)